# ДЗ 4. Операции, axis и broadcasting на данных «Дала Маркет»
Срок: **до понедельника 16.11, 19:00**.
Сдача: папка `hw04` репозитория `dsa-homework` (`hw04.ipynb`): `git add hw04` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем. Небольшой цикл по городам или категориям при сборке таблицы допустим.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import numpy as np
# 1
a = np.array([2, 4, 6]); print(a / 2, a // 4)
# 2
m = np.ones((3, 4)); print(m.sum(axis=1))
# 3
print(np.arange(3) + np.arange(3).reshape(3, 1))
# 4
x = np.array([1, 4, 9]); print(np.sqrt(x).sum())
# 5
print((np.arange(1, 5) ** 2).mean())
```

Ваши ответы и объяснения:

1. [1. 2. 3.] [0 1 1]
оператор / делит поэлементно с плавающей точкой (float) // выполняет целочисленное деление с округлением вниз
2. [4. 4. 4.]
np.ones((3, 4)) создаёт матрицу из единиц размера 3×4 sum(axis=1) суммирует элементы вдоль столбцов по каждой строке давая 3 суммы по 4.0
3. [[0 1 2], [1 2 3], [2 3 4]]
срабатывает broadcasting строка формы (3,) складывается со столбцом формы (3, 1) образуя матрицу 3×3 всех попарных сумм
4. 6.0
np.sqrt(x) извлекает корень поэлементно [1., 2., 3.] .sum() складывает их 1 + 2 + 3 = 6.0
5. 7.5
np.arange(1, 5) создаёт [1, 2, 3, 4] возведение в квадрат даёт [1, 4, 9, 16] .mean() считает среднее (1 + 4 + 9 + 16) / 4 = 7.5

In [1]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [2]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])
month  = np.array([r["order_date"][:7] for r in rows])      # "2026-09" или "2026-10"
cat    = np.array([r["category"] for r in rows])
oid    = np.array([r["order_id"] for r in rows])

print(amount.shape, np.unique(month))

(5463,) ['2026-09' '2026-10']


In [3]:
# Чистые данные: без дублей (оставляем первое вхождение order_id), без TEST, только оплаченные
_, first = np.unique(oid, return_index=True)
keep = np.zeros(len(oid), dtype=bool)
keep[first] = True
clean = keep & (cust != "TEST") & paid
print("строк всего:", len(oid), "| в чистых данных:", clean.sum())

строк всего: 5463 | в чистых данных: 4739


## [База] 1. Выручка по чистым данным
Сколько строк осталось в `clean`? Выручка за сентябрь и октябрь и изменение в %. Совпадает ли с ДЗ 2?

In [4]:
# 1
sep_rev = amount[clean & (month == "2026-09")].sum()
oct_rev = amount[clean & (month == "2026-10")].sum()
change_pct = (oct_rev - sep_rev) / sep_rev * 100

print(f"Строк в clean: {clean.sum()}")
print(f"Выручка за сентябрь: {sep_rev:,.0f} ₸")
print(f"Выручка за октябрь:  {oct_rev:,.0f} ₸")
print(f"Изменение: {change_pct:.2f}% (совпадает с ДЗ 2: 35 243 410 ₸, 34 419 520 ₸ и -2.34%)")

Строк в clean: 4739
Выручка за сентябрь: 35,243,410 ₸
Выручка за октябрь:  34,419,520 ₸
Изменение: -2.34% (совпадает с ДЗ 2: 35 243 410 ₸, 34 419 520 ₸ и -2.34%)


В чистых данных осталось 4739 заказов, выручка снизилась с 35243410 в сентябре до 34419520 в октябре (-2.34%), это полностью совпадает с расчетом из ДЗ 2, поэтому реальная просадка небольшая.

## [База] 2. Таблица «категория × месяц»
Соберите таблицу 5 × 2 по образцу таблицы городов с урока (категории: `np.unique(cat)`).
Какая категория приносит больше всего выручки за два месяца (`axis`)? Какая упала сильнее всех в %?

In [5]:
# 2
cats = np.unique(cat)
months = np.unique(month)
cat_table = np.array([[amount[clean & (cat == c) & (month == m)].sum() for m in months] for c in cats])

cat_totals = cat_table.sum(axis=1)
best_idx = cat_totals.argmax()
cat_pct = (cat_table[:, 1] - cat_table[:, 0]) / cat_table[:, 0] * 100
worst_idx = cat_pct.argmin()

for i, c in enumerate(cats):
    print(f"{c:12s} | Сентябрь: {cat_table[i, 0]:10,.0f} ₸ | Октябрь: {cat_table[i, 1]:10,.0f} ₸ | Всего: {cat_totals[i]:10,.0f} ₸ | Изм.: {cat_pct[i]:+6.2f}%")

print(f"\nЛидер по выручке за 2 месяца: {cats[best_idx]} ({cat_totals[best_idx]:,.0f} ₸)")
print(f"Сильнее всех упала в %: {cats[worst_idx]} ({cat_pct[worst_idx]:.2f}%)")

Дом          | Сентябрь:  4,997,730 ₸ | Октябрь:  5,125,790 ₸ | Всего: 10,123,520 ₸ | Изм.:  +2.56%
Книги        | Сентябрь:  1,065,300 ₸ | Октябрь:  1,140,910 ₸ | Всего:  2,206,210 ₸ | Изм.:  +7.10%
Красота      | Сентябрь:  2,610,950 ₸ | Октябрь:  2,678,990 ₸ | Всего:  5,289,940 ₸ | Изм.:  +2.61%
Одежда       | Сентябрь:  9,167,100 ₸ | Октябрь:  8,505,730 ₸ | Всего: 17,672,830 ₸ | Изм.:  -7.21%
Электроника  | Сентябрь: 17,402,330 ₸ | Октябрь: 16,968,100 ₸ | Всего: 34,370,430 ₸ | Изм.:  -2.50%

Лидер по выручке за 2 месяца: Электроника (34,370,430 ₸)
Сильнее всех упала в %: Одежда (-7.21%)


Больше всего выручки за два месяца приносит Электроника (34370430, почти половина оборота), а сильнее всех в процентах упала Одежда (-7.21%), поэтому стоит обратить внимание на продажи в этой категории.

## [База] 3. Средний чек по категориям в октябре
Соберите вторую таблицу: **число** заказов по категориям и месяцам (сумма маски, а не сумма amount).
Средний чек = выручка / число заказов, одной операцией над двумя таблицами. У какой категории он самый высокий?

In [6]:
# 3
cat_counts = np.array([[(clean & (cat == c) & (month == m)).sum() for m in months] for c in cats])
avg_check = cat_table / cat_counts
oct_best_idx = avg_check[:, 1].argmax()

for i, c in enumerate(cats):
    print(f"{c:12s} | Заказов (сен/окт): {cat_counts[i, 0]:3d}/{cat_counts[i, 1]:3d} | Ср. чек окт: {avg_check[i, 1]:9,.2f} ₸")

print(f"\nСамый высокий средний чек в октябре: {cats[oct_best_idx]} ({avg_check[oct_best_idx, 1]:,.2f} ₸)")

Дом          | Заказов (сен/окт): 476/476 | Ср. чек окт: 10,768.47 ₸
Книги        | Заказов (сен/окт): 265/299 | Ср. чек окт:  3,815.75 ₸
Красота      | Заказов (сен/окт): 422/435 | Ср. чек окт:  6,158.60 ₸
Одежда       | Заказов (сен/окт): 678/652 | Ср. чек окт: 13,045.60 ₸
Электроника  | Заказов (сен/окт): 531/505 | Ср. чек окт: 33,600.20 ₸

Самый высокий средний чек в октябре: Электроника (33,600.20 ₸)


Самый высокий средний чек в октябре у категории Электроника (33600.20), он в несколько раз выше остальных категорий, поэтому каждый такой заказ сильно влияет на выручку.

## [Стандарт] 4. Доля городов в выручке месяца
Используйте `table` городов с урока. Доля каждого города в выручке своего месяца: `table / table.sum(axis=0)`.
Проверьте, что каждый столбец в сумме даёт 1. Как изменилась доля Шымкента?

In [7]:
# 4
cities = np.unique(city)
table = np.array([[amount[clean & (city == c) & (month == m)].sum() for m in months] for c in cities])
city_share = table / table.sum(axis=0)

for i, c in enumerate(cities):
    print(f"{c:10s} | Сентябрь: {city_share[i, 0]*100:5.2f}% | Октябрь: {city_share[i, 1]*100:5.2f}%")

print("Сумма долей по столбцам:", city_share.sum(axis=0))
shym_share = city_share[cities == "Шымкент"][0]
shym_diff_pp = (shym_share[1] - shym_share[0]) * 100
print(f"Доля Шымкента: {shym_share[0]*100:.2f}% -> {shym_share[1]*100:.2f}% ({shym_diff_pp:+.2f} п.п.)")

Актобе     | Сентябрь:  7.31% | Октябрь:  7.38%
Алматы     | Сентябрь: 41.15% | Октябрь: 42.54%
Астана     | Сентябрь: 25.63% | Октябрь: 25.71%
Атырау     | Сентябрь:  2.62% | Октябрь:  2.82%
Караганда  | Сентябрь: 10.35% | Октябрь:  9.80%
Шымкент    | Сентябрь: 12.93% | Октябрь: 11.75%
Сумма долей по столбцам: [1. 1.]
Доля Шымкента: 12.93% -> 11.75% (-1.19 п.п.)


Доля Шымкента в общей выручке снизилась с 12.93% в сентябре до 11.75% в октябре (-1.19 п.п.), тогда как Алматы и Астана свою долю сохранили и немного увеличили.

## [Стандарт] 5. Кто дал падение
Вклад каждого города в общее падение выручки: изменение города / общее изменение. Сколько процентов падения даёт Шымкент?
Напишите менеджеру одну фразу: где искать причину.

In [8]:
# 5
city_diff = table[:, 1] - table[:, 0]
total_diff = city_diff.sum()
drop_contrib = city_diff / total_diff * 100

for i, c in enumerate(cities):
    print(f"{c:10s} | Изменение: {city_diff[i]:+10,.0f} ₸ | Вклад в общее падение: {drop_contrib[i]:+6.2f}%")

shym_contrib = drop_contrib[cities == "Шымкент"][0]
print(f"\nОбщее изменение выручки: {total_diff:,.0f} ₸")
print(f"Вклад Шымкента в падение: {shym_contrib:.2f}%")

Актобе     | Изменение:    -36,540 ₸ | Вклад в общее падение:  +4.44%
Алматы     | Изменение:   +141,950 ₸ | Вклад в общее падение: -17.23%
Астана     | Изменение:   -183,690 ₸ | Вклад в общее падение: +22.30%
Атырау     | Изменение:    +45,870 ₸ | Вклад в общее падение:  -5.57%
Караганда  | Изменение:   -276,840 ₸ | Вклад в общее падение: +33.60%
Шымкент    | Изменение:   -514,640 ₸ | Вклад в общее падение: +62.46%

Общее изменение выручки: -823,890 ₸
Вклад Шымкента в падение: 62.46%


Шымкент дает 62.46% всего падения выручки (-514640 из общих -823890), причину нужно искать в доставке и логистике Шымкента, где больше всего отмен и долгие сроки доставки.

## [Стандарт] 6. Исправьте код от AI
AI написал код «доля сентября и октября внутри каждого города». Код падает. Объясните, почему, и исправьте.
```python
share = table / table.sum(axis=1)
```

In [9]:
# 6
# У table размер (6, 2), а у table.sum(axis=1) размер (6,), поэтому NumPy не может их разделить по правилам broadcasting
# Нужно добавить keepdims=True чтобы сумма строк имела размер (6, 1)
share = table / table.sum(axis=1, keepdims=True)

for i, c in enumerate(cities):
    print(f"{c:10s} | Сентябрь: {share[i, 0]*100:5.2f}% | Октябрь: {share[i, 1]*100:5.2f}% | Сумма: {share[i].sum():.1f}")

Актобе     | Сентябрь: 50.36% | Октябрь: 49.64% | Сумма: 1.0
Алматы     | Сентябрь: 49.76% | Октябрь: 50.24% | Сумма: 1.0
Астана     | Сентябрь: 50.51% | Октябрь: 49.49% | Сумма: 1.0
Атырау     | Сентябрь: 48.79% | Октябрь: 51.21% | Сумма: 1.0
Караганда  | Сентябрь: 51.97% | Октябрь: 48.03% | Сумма: 1.0
Шымкент    | Сентябрь: 52.99% | Октябрь: 47.01% | Сумма: 1.0


Код падал потому что размеры (6, 2) и (6,) не совпадают при делении, а после исправления через keepdims=True видно, что внутри Шымкента доля октября просела сильнее всего (47.01% против 52.99% в сентябре).

## [Челлендж] 7. Стандартизация суммы заказа
Для чистых оплаченных заказов посчитайте z-оценку: `z = (x - x.mean()) / x.std()`.
Сколько заказов с |z| > 3 и какая это доля? Проверьте, что у `z` среднее ≈ 0 и std = 1.

In [10]:
# 7
x = amount[clean]
z = (x - x.mean()) / x.std()
outliers_mask = np.abs(z) > 3
outliers_count = outliers_mask.sum()
outliers_share = outliers_mask.mean() * 100

print(f"Заказов с |z| > 3: {outliers_count} ({outliers_share:.2f}% от чистых заказов)")
print(f"Проверка z: mean = {z.mean():.2e} (≈ 0), std = {z.std():.6f} (= 1)")

Заказов с |z| > 3: 87 (1.84% от чистых заказов)
Проверка z: mean = 7.05e-17 (≈ 0), std = 1.000000 (= 1)


Среди чистых заказов 87 штук (1.84%) имеют z больше 3 по модулю (от 54740), почти все из них это дорогие заказы электроники, которые тянут среднее вверх.

## [Челлендж] 8. Масштабирование признаков (заготовка для ML)
Соберите матрицу `X` из двух столбцов: `amount` и `days` для чистых заказов без пропусков в `days` (`np.column_stack`).
Приведите каждый столбец к диапазону [0, 1] одной строкой: `(X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))`.
Проверьте min и max каждого столбца. Объясните своими словами, зачем модели нужны признаки в одном масштабе.

In [11]:
# 8
valid = clean & ~np.isnan(days)
X = np.column_stack([amount[valid], days[valid]])
X_scaled = (X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))

print("Форма матрицы X:", X_scaled.shape)
print("Min по столбцам:", X_scaled.min(axis=0))
print("Max по столбцам:", X_scaled.max(axis=0))

Форма матрицы X: (4595, 2)
Min по столбцам: [0. 0.]
Max по столбцам: [1. 1.]


Оба признака (4595 заказов без пропусков срока доставки) приведены к диапазону от 0 до 1. Одинаковый масштаб нужен модели чтобы сумма заказа в десятках тысяч тенге не перевешивала срок доставки в днях при обучении.